# Clase 8 — Práctica: QAOA y Max-Cut

En la clase teórica entrenaste un circuito QAOA con una sola repetición del ciclo coste-mezcla (`p = 1`) sobre un ciclo de seis nodos. Ahora comprueba tú mismo si añadir una segunda repetición (`p = 2`) mejora la calidad del corte que obtienes.

## Enunciado

Cambia `p = 1` por `p = 2` en el circuito QAOA de la clase teórica. Con dos repeticiones necesitas dos símbolos de coste y dos de mezcla, y dos llamadas a `tfq.util.exponential` encadenadas con `+=`, una por cada repetición del ciclo. Entrena de nuevo el modelo y compara el corte medio obtenido con el de `p = 1`. Anota si la solución mejora, y relaciona el resultado con la idea de que más repeticiones acercan la solución al óptimo.

**Pistas**

- Declara cuatro símbolos con `sympy.symbols("gamma1 gamma2 beta1 beta2")`
- La primera llamada a `tfq.util.exponential` usa `gamma1` y `beta1`; la segunda usa `gamma2` y `beta2`; encadénalas con `circuito_qaoa += ...` dos veces
- El resto del pipeline (modelo, `compile`, `fit`, muestreo, comparación con fuerza bruta) es idéntico al de `p = 1`, solo cambian el circuito y la lista de símbolos que pasas a `tfq.layers.Sample`
- Reutiliza el mismo grafo (ciclo de seis nodos) y el mismo `hamiltoniano_coste` / `hamiltoniano_mezcla` que ya calculaste en la clase teórica

In [ ]:
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import itertools

# Mismo grafo que en la clase teórica: ciclo de seis nodos
grafo = nx.cycle_graph(6)
qubits = [cirq.GridQubit(0, i) for i in range(grafo.number_of_nodes())]

estado_inicial = cirq.Circuit()
for qubit in qubits:
    estado_inicial.append(cirq.H(qubit))

hamiltoniano_mezcla = sum(cirq.PauliString(cirq.X(qubits[i])) for i in range(len(qubits)))

hamiltoniano_coste = grafo.number_of_edges() / 2
for nodo_a, nodo_b in grafo.edges():
    hamiltoniano_coste += cirq.PauliString(0.5 * cirq.Z(qubits[nodo_a]) * cirq.Z(qubits[nodo_b]))

p = 2  # ahora con dos repeticiones del ciclo coste-mezcla

# TODO: declara los cuatro símbolos entrenables con sympy.symbols("gamma1 gamma2 beta1 beta2")

circuito_qaoa = cirq.Circuit()

# TODO: encadena dos llamadas a tfq.util.exponential con circuito_qaoa +=
#       la primera con operators=[hamiltoniano_coste, hamiltoniano_mezcla] y coefficients=[gamma1, beta1]
#       la segunda con coefficients=[gamma2, beta2]

print(circuito_qaoa)

In [ ]:
entradas = tfq.convert_to_tensor([estado_inicial])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
salida = tfq.layers.PQC(circuito_qaoa, hamiltoniano_coste)(capa_entrada)

modelo_qaoa_p2 = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)
modelo_qaoa_p2.compile(loss=tf.keras.losses.MAE, optimizer=tf.keras.optimizers.Adam())

# TODO: entrena el modelo con .fit(entradas, np.array([0]), epochs=200) y guarda el resultado en "historial"

plt.plot(historial.history["loss"])
plt.title("QAOA con p=2 — pérdida durante el entrenamiento")
plt.xlabel("Iteración")
plt.ylabel("Pérdida")
plt.show()

In [ ]:
parametros_entrenados = modelo_qaoa_p2.trainable_variables
circuito_con_datos = tfq.layers.AddCircuit()(entradas, append=circuito_qaoa)

# TODO: muestrea el circuito entrenado con tfq.layers.Sample, pasando
#       symbol_names=[gamma1, gamma2, beta1, beta2], symbol_values=parametros_entrenados y repetitions=1000
#       guarda el resultado en "muestras"

cortes_obtenidos_p2 = []
for bits in muestras.values:
    grupo_uno = [i for i, bit in enumerate(bits) if bit == 1]
    cortes_obtenidos_p2.append(nx.algorithms.cuts.cut_size(grafo, grupo_uno))

corte_maximo = max(
    nx.algorithms.cuts.cut_size(grafo, list(combo))
    for r in range(grafo.number_of_nodes() + 1)
    for combo in itertools.combinations(grafo.nodes(), r)
)

# TODO: imprime el corte medio obtenido con p=2 (np.mean(cortes_obtenidos_p2)) y el corte máximo,
#       y compáralo con el corte medio que obtuviste en la clase teórica con p=1

<details>
<summary>💡 Ver solución</summary>

```python
import tensorflow as tf
import tensorflow_quantum as tfq
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import itertools

# Mismo grafo que en la clase teórica: ciclo de seis nodos
grafo = nx.cycle_graph(6)
qubits = [cirq.GridQubit(0, i) for i in range(grafo.number_of_nodes())]

estado_inicial = cirq.Circuit()
for qubit in qubits:
    estado_inicial.append(cirq.H(qubit))

hamiltoniano_mezcla = sum(cirq.PauliString(cirq.X(qubits[i])) for i in range(len(qubits)))

hamiltoniano_coste = grafo.number_of_edges() / 2
for nodo_a, nodo_b in grafo.edges():
    hamiltoniano_coste += cirq.PauliString(0.5 * cirq.Z(qubits[nodo_a]) * cirq.Z(qubits[nodo_b]))

p = 2  # ahora con dos repeticiones del ciclo coste-mezcla

# Cuatro símbolos entrenables: un gamma y un beta por cada repetición
gamma1, gamma2, beta1, beta2 = sympy.symbols("gamma1 gamma2 beta1 beta2")

circuito_qaoa = cirq.Circuit()
# Primera repetición del ciclo coste-mezcla
circuito_qaoa += tfq.util.exponential(
    operators=[hamiltoniano_coste, hamiltoniano_mezcla],
    coefficients=[gamma1, beta1],
)
# Segunda repetición del ciclo coste-mezcla
circuito_qaoa += tfq.util.exponential(
    operators=[hamiltoniano_coste, hamiltoniano_mezcla],
    coefficients=[gamma2, beta2],
)

print(circuito_qaoa)

entradas = tfq.convert_to_tensor([estado_inicial])
capa_entrada = tf.keras.layers.Input(shape=(), dtype=tf.dtypes.string)
salida = tfq.layers.PQC(circuito_qaoa, hamiltoniano_coste)(capa_entrada)

modelo_qaoa_p2 = tf.keras.models.Model(inputs=capa_entrada, outputs=salida)
modelo_qaoa_p2.compile(loss=tf.keras.losses.MAE, optimizer=tf.keras.optimizers.Adam())

historial = modelo_qaoa_p2.fit(entradas, np.array([0]), epochs=200)

plt.plot(historial.history["loss"])
plt.title("QAOA con p=2 — pérdida durante el entrenamiento")
plt.xlabel("Iteración")
plt.ylabel("Pérdida")
plt.show()

parametros_entrenados = modelo_qaoa_p2.trainable_variables
circuito_con_datos = tfq.layers.AddCircuit()(entradas, append=circuito_qaoa)
muestras = tfq.layers.Sample()(
    circuito_con_datos,
    symbol_names=[gamma1, gamma2, beta1, beta2],
    symbol_values=parametros_entrenados,
    repetitions=1000,
)

cortes_obtenidos_p2 = []
for bits in muestras.values:
    grupo_uno = [i for i, bit in enumerate(bits) if bit == 1]
    cortes_obtenidos_p2.append(nx.algorithms.cuts.cut_size(grafo, grupo_uno))

corte_maximo = max(
    nx.algorithms.cuts.cut_size(grafo, list(combo))
    for r in range(grafo.number_of_nodes() + 1)
    for combo in itertools.combinations(grafo.nodes(), r)
)

print("Corte medio obtenido con p=2:", np.mean(cortes_obtenidos_p2))
print("Mejor corte posible (fuerza bruta):", corte_maximo)

# Compara este resultado con el corte medio que obtuviste en la clase teórica con p=1:
# con p=2 el circuito tiene el doble de parámetros y el doble de profundidad, así que
# el corte medio debería acercarse más al máximo teórico (aunque el margen de mejora
# es pequeño en un grafo tan sencillo como este ciclo de seis nodos).
```

</details>